# NB14 토큰 표준 — ERC-20·ERC-721 장부 흉내, Keccak-256 과 인터페이스 식별자

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB14-1: ERC-20 장부 흉내 — 잔액·허용량 사전, 여섯 함수, 두 이벤트, 그리고 approve 경쟁 조건 (공부용)
class ERC20Toy:
    """EIP-20 의 필수 함수만. 실패는 예외로 던진다(표준의 SHOULD throw). 이벤트는 self.events 에 적는다."""
    def __init__(self, name, symbol):
        self.name, self.symbol = name, symbol             # OPTIONAL 항목 — 있어도 되고 없어도 된다
        self.balances = {}                                 # 「모든 주소는 처음부터 0」 — 사전에 없으면 0으로 읽는다
        self.allowances = {}                               # (owner, spender) -> 남은 허용량
        self.events = []
    def totalSupply(self):          return sum(self.balances.values())
    def balanceOf(self, owner):     return self.balances.get(owner, 0)
    def allowance(self, o, s):      return self.allowances.get((o, s), 0)
    def mint(self, to, value):                            # 표준 밖 — 발행 규칙은 발행자 몫. 0x0 에서 온 Transfer 이벤트만 권고(SHOULD)
        self.balances[to] = self.balanceOf(to) + value
        self.events.append(("Transfer", "0x0", to, value))
    def transfer(self, caller, to, value):
        if self.balanceOf(caller) < value:
            raise ValueError(f"잔액 부족: {caller} 는 {self.balanceOf(caller)} 뿐")
        self.balances[caller] -= value
        self.balances[to] = self.balanceOf(to) + value
        self.events.append(("Transfer", caller, to, value))   # 0 수량이라도 이벤트는 낸다(MUST)
        return True
    def approve(self, caller, spender, value):
        self.allowances[(caller, spender)] = value            # 「다시 부르면 덮어쓴다」
        self.events.append(("Approval", caller, spender, value))
        return True
    def transferFrom(self, caller, frm, to, value):
        if self.allowance(frm, caller) < value:
            raise ValueError(f"허용량 부족: {frm} 이 {caller} 에게 허용한 양은 {self.allowance(frm, caller)}")
        if self.balanceOf(frm) < value:
            raise ValueError("잔액 부족")
        self.allowances[(frm, caller)] -= value
        self.balances[frm] -= value
        self.balances[to] = self.balanceOf(to) + value
        self.events.append(("Transfer", frm, to, value))
        return True

tok = ERC20Toy("Toy Token", "TOY")
tok.mint("Alice", 1000)
tok.transfer("Alice", "Bob", 250)
print("잔액 — Alice:", tok.balanceOf("Alice"), "Bob:", tok.balanceOf("Bob"), "Carol(처음 보는 주소):", tok.balanceOf("Carol"), "| 총량:", tok.totalSupply())
try:
    tok.transfer("Bob", "Carol", 300)
except ValueError as e:
    print("Bob → Carol 300:", e)

# approve / transferFrom — 계약(여기서는 'Exchange')이 Alice 대신 토큰을 가져가는 인출 흐름
tok.approve("Alice", "Exchange", 100)
tok.transferFrom("Exchange", "Alice", "Dave", 60)
print("Exchange 가 Alice 에게서 60 인출 → Dave:", tok.balanceOf("Dave"), "| 남은 허용량:", tok.allowance("Alice", "Exchange"))

# approve 경쟁 조건 — 표준이 「먼저 0 으로 두라」고 권고한 이유를 순서로 재연한다
tok2 = ERC20Toy("Toy", "TOY"); tok2.mint("Alice", 1000)
tok2.approve("Alice", "Spender", 100)                      # ① Alice: Spender 에게 100 허용
# Alice 가 「100 은 너무 많다, 50 으로 줄이자」며 approve(Spender, 50) 를 보낸다. 그런데 블록에 들어가는 순서는 보낸 순서가 아니다.
tok2.transferFrom("Spender", "Alice", "Spender", 100)      # ② Spender 가 Alice 의 거래보다 먼저 100 을 전부 꺼낸다
tok2.approve("Alice", "Spender", 50)                       # ③ 그 뒤 Alice 의 approve(50) 이 적용된다 — 「덮어쓰기」라 50 이 새로 열린다
tok2.transferFrom("Spender", "Alice", "Spender", 50)       # ④ Spender 가 50 을 더 꺼낸다
print("경쟁 조건 — Alice 는 50 만 허용할 뜻이었으나 Spender 가 가져간 양:", tok2.balanceOf("Spender"), "(= 100 + 50)")
print("이벤트 수:", len(tok2.events), "| 마지막 셋:", tok2.events[-3:])

잔액 — Alice: 750 Bob: 250 Carol(처음 보는 주소): 0 | 총량: 1000
Bob → Carol 300: 잔액 부족: Bob 는 250 뿐
Exchange 가 Alice 에게서 60 인출 → Dave: 60 | 남은 허용량: 40
경쟁 조건 — Alice 는 50 만 허용할 뜻이었으나 Spender 가 가져간 양: 150 (= 100 + 50)
이벤트 수: 5 | 마지막 셋: [('Transfer', 'Alice', 'Spender', 100), ('Approval', 'Alice', 'Spender', 50), ('Transfer', 'Alice', 'Spender', 50)]


In [ ]:
# NB14-2: ERC-721 장부 흉내 — 번호 → 소유자, 승인·운영자, safeTransferFrom 의 수취인 검사, 가변 URI (공부용)
MAGIC = "onERC721Received-magic"                           # 실제로는 bytes4(keccak256("onERC721Received(address,address,uint256,bytes)")) — 아래 NB14-3 에서 계산

class Wallet:                                              # 「계약 수취인」 흉내. 받을 준비가 된 계약은 마법값을 돌려준다
    def __init__(self, ready): self.ready = ready
    def onERC721Received(self, operator, frm, tokenId, data): return MAGIC if self.ready else "something else"

class ERC721Toy:
    def __init__(self):
        self.owner = {}            # tokenId -> owner
        self.approved = {}         # tokenId -> approved address
        self.operators = set()     # (owner, operator)
        self.uri = {}              # tokenId -> URI 문자열 (메타데이터 확장, OPTIONAL)
        self.events = []
    def ownerOf(self, tid):
        if tid not in self.owner: raise KeyError(f"유효하지 않은 NFT {tid}")   # 유효하지 않은 NFT(0 주소 배정 등)의 조회는 실패한다 — 원문 ownerOf 주석의 요지
        return self.owner[tid]
    def balanceOf(self, who):  return sum(1 for o in self.owner.values() if o == who)
    def mint(self, to, tid, uri):                           # 표준 밖 — 생성 규칙은 발행자 몫
        self.owner[tid] = to; self.uri[tid] = uri; self.events.append(("Transfer", "0x0", to, tid))
    def tokenURI(self, tid):   self.ownerOf(tid); return self.uri[tid]
    def _authorized(self, caller, tid):
        o = self.ownerOf(tid)
        return caller == o or self.approved.get(tid) == caller or (o, caller) in self.operators
    def approve(self, caller, approved, tid):
        o = self.ownerOf(tid)
        if not (caller == o or (o, caller) in self.operators): raise PermissionError("소유자나 운영자만 승인할 수 있다")
        self.approved[tid] = approved; self.events.append(("Approval", o, approved, tid))
    def setApprovalForAll(self, caller, operator, ok):
        (self.operators.add if ok else self.operators.discard)((caller, operator))
        self.events.append(("ApprovalForAll", caller, operator, ok))
    def transferFrom(self, caller, frm, to, tid):
        if not self._authorized(caller, tid): raise PermissionError(f"{caller} 는 {tid} 를 옮길 권한이 없다")
        if self.owner[tid] != frm: raise ValueError("_from 이 현재 소유자가 아니다")
        if to == "0x0": raise ValueError("0 주소로는 보낼 수 없다")
        self.owner[tid] = to; self.approved.pop(tid, None)   # 「승인은 이전 시 초기화된다」
        self.events.append(("Transfer", frm, to, tid))
    def safeTransferFrom(self, caller, frm, to, tid, data=b"", contracts=None):
        self.transferFrom(caller, frm, to, tid)
        if contracts and to in contracts:                      # 수취인이 계약이면 받을 준비를 묻는다
            if contracts[to].onERC721Received(caller, frm, tid, data) != MAGIC:
                self.owner[tid] = frm; self.events.pop()       # 되돌림(실제로는 거래 전체가 revert)
                raise ValueError(f"수취인 {to} 가 마법값을 돌려주지 않아 이전을 되돌렸다")

nft = ERC721Toy()
nft.mint("Alice", 7, "https://example.invalid/meta/7.json")
print("7번 소유자:", nft.ownerOf(7), "| tokenURI:", nft.tokenURI(7))
contracts = {"ReadyContract": Wallet(ready=True), "DumbContract": Wallet(ready=False)}
try:
    nft.safeTransferFrom("Alice", "Alice", "DumbContract", 7, contracts=contracts)
except ValueError as e:
    print("safeTransferFrom → 준비 안 된 계약:", e, "| 소유자는 여전히", nft.ownerOf(7))
nft.transferFrom("Alice", "Alice", "DumbContract", 7)       # 「안전하지 않은」 이전은 묻지 않고 보낸다 — 표준의 대문자 경고가 이 자리
print("transferFrom → 같은 계약: 소유자", nft.ownerOf(7), "(꺼낼 코드가 없는 계약이면 영구히 잠긴다)")

nft.mint("Bob", 8, "https://example.invalid/meta/8.json")
nft.setApprovalForAll("Bob", "Marketplace", True)           # Bob 이 장터 계약을 운영자로 둔다
nft.transferFrom("Marketplace", "Bob", "Carol", 8)           # 운영자가 Bob 의 토큰을 Carol 에게 옮긴다
print("운영자 이전 뒤 8번 소유자:", nft.ownerOf(8), "| Bob 의 보유 수:", nft.balanceOf("Bob"), "| Carol:", nft.balanceOf("Carol"))

nft.uri[8] = "https://example.invalid/meta/8-v2.json"       # 발행자가 URI 를 바꾼다 — 소유자 칸은 그대로
print("URI 를 바꾼 뒤 — 소유자:", nft.ownerOf(8), "| tokenURI:", nft.tokenURI(8), "(체인이 보증하는 것은 왼쪽, 바뀔 수 있는 것은 오른쪽)")

7번 소유자: Alice | tokenURI: https://example.invalid/meta/7.json
safeTransferFrom → 준비 안 된 계약: 수취인 DumbContract 가 마법값을 돌려주지 않아 이전을 되돌렸다 | 소유자는 여전히 Alice
transferFrom → 같은 계약: 소유자 DumbContract (꺼낼 코드가 없는 계약이면 영구히 잠긴다)
운영자 이전 뒤 8번 소유자: Carol | Bob 의 보유 수: 0 | Carol: 1
URI 를 바꾼 뒤 — 소유자: Carol | tokenURI: https://example.invalid/meta/8-v2.json (체인이 보증하는 것은 왼쪽, 바뀔 수 있는 것은 오른쪽)


In [ ]:
# NB14-3: Keccak-256 을 손으로 — 채움 바이트 하나가 SHA3-256 과 Keccak-256 을 가른다. 그 다음 ERC-165/721 의 식별자를 재계산 (공부용, 실사용 금지)
import hashlib
from functools import reduce

RC = [0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000, 0x000000000000808B, 0x0000000080000001,
      0x8000000080008081, 0x8000000000008009, 0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
      0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003, 0x8000000000008002, 0x8000000000000080,
      0x000000000000800A, 0x800000008000000A, 0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008]
ROT = [[0, 36, 3, 41, 18], [1, 44, 10, 45, 2], [62, 6, 43, 15, 61], [28, 55, 25, 21, 56], [27, 20, 39, 8, 14]]   # 회전량 r[x][y]
M64 = (1 << 64) - 1
def rol(v, n): return ((v << n) | (v >> (64 - n))) & M64 if n else v

def keccak_f(A):                                           # A[x][y]: 5×5 개의 64비트 레인. 24라운드 θ·ρ·π·χ·ι
    for rc in RC:
        C = [A[x][0] ^ A[x][1] ^ A[x][2] ^ A[x][3] ^ A[x][4] for x in range(5)]
        D = [C[(x - 1) % 5] ^ rol(C[(x + 1) % 5], 1) for x in range(5)]
        A = [[A[x][y] ^ D[x] for y in range(5)] for x in range(5)]                       # θ
        B = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                B[y][(2 * x + 3 * y) % 5] = rol(A[x][y], ROT[x][y])                      # ρ, π
        A = [[B[x][y] ^ ((~B[(x + 1) % 5][y]) & B[(x + 2) % 5][y]) for y in range(5)] for x in range(5)]   # χ
        A[0][0] ^= rc                                                                    # ι
    return A

def sponge256(msg: bytes, pad_byte: int) -> bytes:        # 용량 512비트 → 속도(rate) 1088비트 = 136바이트, 출력 256비트
    rate = 136
    padded = bytearray(msg) + bytearray([pad_byte]) + bytearray((-len(msg) - 2) % rate) + bytearray([0x80])
    if len(msg) % rate == rate - 1:                        # 한 바이트만 남는 경우: pad_byte | 0x80 한 바이트로 채운다
        padded = bytearray(msg) + bytearray([pad_byte | 0x80])
    A = [[0] * 5 for _ in range(5)]
    for off in range(0, len(padded), rate):
        block = padded[off:off + rate]
        for i in range(rate // 8):                          # 레인 i → (x, y) = (i % 5, i // 5), 리틀엔디안
            A[i % 5][i // 5] ^= int.from_bytes(block[8 * i:8 * i + 8], "little")
        A = keccak_f(A)
    return b"".join(A[i % 5][i // 5].to_bytes(8, "little") for i in range(4))

def sha3_256_manual(m): return sponge256(m, 0x06)          # NIST SHA-3: 채움 접미 0x06
def keccak256(m):       return sponge256(m, 0x01)          # 이더리움 Keccak-256: 채움 접미 0x01

for m in [b"", b"abc", b"a" * 200]:
    print(f"{m[:12]!r:16} 손 SHA3-256 == hashlib.sha3_256 ? {sha3_256_manual(m) == hashlib.sha3_256(m).digest()}")
print("keccak256(b'')    =", keccak256(b"").hex())
print("sha3_256(b'')     =", hashlib.sha3_256(b"").hexdigest(), "(채움 바이트 하나가 다를 뿐인데 전혀 다른 값)")

def selector(sig: str) -> int: return int.from_bytes(keccak256(sig.encode())[:4], "big")
print()
print("transfer(address,uint256) 선택자 =", f"0x{selector('transfer(address,uint256)'):08x}")
print("supportsInterface(bytes4) 선택자 =", f"0x{selector('supportsInterface(bytes4)'):08x}", "(ERC-165 는 이 값을 식별자로 쓴다)")

erc721 = ["balanceOf(address)", "ownerOf(uint256)", "safeTransferFrom(address,address,uint256,bytes)",
          "safeTransferFrom(address,address,uint256)", "transferFrom(address,address,uint256)", "approve(address,uint256)",
          "setApprovalForAll(address,bool)", "getApproved(uint256)", "isApprovedForAll(address,address)"]
erc721_meta = ["name()", "symbol()", "tokenURI(uint256)"]
xor = lambda sigs: reduce(lambda a, b: a ^ b, (selector(s) for s in sigs))
print(f"ERC-721 식별자(9 함수 XOR)        = 0x{xor(erc721):08x}   표준 원문: 0x80ac58cd → 일치 {xor(erc721) == 0x80ac58cd}")
print(f"ERC-721 Metadata 식별자(3 함수)   = 0x{xor(erc721_meta):08x}   표준 원문: 0x5b5e139f → 일치 {xor(erc721_meta) == 0x5b5e139f}")
print(f"onERC721Received 마법값           = 0x{selector('onERC721Received(address,address,uint256,bytes)'):08x}   표준 원문: 0x150b7a02 → 일치 {selector('onERC721Received(address,address,uint256,bytes)') == 0x150b7a02}")

b''              손 SHA3-256 == hashlib.sha3_256 ? True
b'abc'           손 SHA3-256 == hashlib.sha3_256 ? True
b'aaaaaaaaaaaa'  손 SHA3-256 == hashlib.sha3_256 ? True
keccak256(b'')    = c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470
sha3_256(b'')     = a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a (채움 바이트 하나가 다를 뿐인데 전혀 다른 값)

transfer(address,uint256) 선택자 = 0xa9059cbb
supportsInterface(bytes4) 선택자 = 0x01ffc9a7 (ERC-165 는 이 값을 식별자로 쓴다)
ERC-721 식별자(9 함수 XOR)        = 0x80ac58cd   표준 원문: 0x80ac58cd → 일치 True
ERC-721 Metadata 식별자(3 함수)   = 0x5b5e139f   표준 원문: 0x5b5e139f → 일치 True
onERC721Received 마법값           = 0x150b7a02   표준 원문: 0x150b7a02 → 일치 True
